In [1]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: c:\Users\HP\Desktop\Projects\NodeRAG\graphs
The root directory is: c:\Users\HP\Desktop\Projects\NodeRAG


In [2]:
import sys
sys.path.append(root_path)
from graphs.Node import Node

In [3]:
import pickle
import numpy as np
with open(f"{root_path}/graphs/data/graphs/G6_entity_and_chunk_resolution_graph.pkl", "rb") as f:
    medical_g6 = pickle.load(f)
medical_g6

{'medical-0-S-0': <graphs.Node.Node at 0x294da88dc00>,
 'medical-0-S-0-R-0': <graphs.Node.Node at 0x294f1bdd810>,
 'medical-0-S-0-R-1': <graphs.Node.Node at 0x294f1bdd840>,
 'medical-0-S-0-R-2': <graphs.Node.Node at 0x294f1bdd870>,
 'medical-0-S-0-R-3': <graphs.Node.Node at 0x294f1bdd8a0>,
 'medical-0-S-0-R-4': <graphs.Node.Node at 0x294f1bdd8d0>,
 'medical-0-S-1': <graphs.Node.Node at 0x294f1bdd900>,
 'medical-0-S-1-R-0': <graphs.Node.Node at 0x294f1bddd20>,
 'medical-0-S-1-R-1': <graphs.Node.Node at 0x294f1bddd50>,
 'medical-0-S-1-R-2': <graphs.Node.Node at 0x294f1bddd80>,
 'medical-0-S-1-R-3': <graphs.Node.Node at 0x294f1bdddb0>,
 'medical-0-S-1-R-4': <graphs.Node.Node at 0x294f1bddde0>,
 'medical-0-S-1-R-5': <graphs.Node.Node at 0x294f1bdde10>,
 'medical-0-S-1-R-6': <graphs.Node.Node at 0x294f1bdde40>,
 'medical-0-S-1-R-7': <graphs.Node.Node at 0x294f1bdde70>,
 'medical-0-S-1-R-8': <graphs.Node.Node at 0x294f1bddea0>,
 'medical-0-S-1-R-9': <graphs.Node.Node at 0x294f1bdded0>,
 'med

In [4]:
import heapq

def dijkstra_with_paths(graph, src):
    dist = {src: 0}
    if graph[src].node_type == "N" and isinstance(graph[src].source, set):
        for syn in graph[src].source:
            dist[syn] = 0
    prev = {}
    pq = [(value,key) for key,value in dist.items()]

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in graph[u].edges.items():
            nd = d + 1/w
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    return dist, prev


In [5]:
def reconstruct_path(prev, src, dst):
    path = []
    cur = dst

    while cur != src:
        if cur not in prev:
            return None  # no path
        path.append(cur)
        cur = prev[cur]

    path.append(src)
    path.reverse()
    return path


In [6]:
from tqdm import tqdm
def all_pairs_shortest_paths(graph, entry_ids = None):
    all_paths = {}

    for src in tqdm(graph.keys()):
        if entry_ids and src not in entry_ids:
            continue 
        _, prev = dijkstra_with_paths(graph, src)
        all_paths[src] = {}

        for dst in graph:
            if src == dst:
                all_paths[src][dst] = [src]
            else:
                all_paths[src][dst] = reconstruct_path(prev, src, dst)

    return all_paths


In [7]:
import random
entry_node_id_list = list(medical_g6.keys())
entry_node_id_list = [i for i in entry_node_id_list if medical_g6[i].node_type in ["N", "O"]]

In [8]:
paths = all_pairs_shortest_paths(medical_g6, entry_ids= entry_node_id_list)

 93%|█████████▎| 40117/43233 [1:19:58<20:18,  2.56it/s]     

: 

: 

In [ ]:
l,x1,x2 = 0, None, None
for i in entry_node_id_list:
    for j in entry_node_id_list:
        p = paths[i][j]
        if len(p) >= l:
            x1 = i
            x2 = j
print("Max dist:", l)
print(x1,x2)

In [ ]:
with open("data/paths/paths.pkl", "wb") as f:
    pickle.dump(paths, f)
with open("data/paths/paths.pkl", "rb") as f:
    path_dict = pickle.load(f)
path_dict